# Offline DQN with TextEmbedder

Same FrozenLake offline DQN loop as `02_train_offline.ipynb`, but each step is rendered with `TextEmbedder`: `action` is `type: "token"` (integer id → one `embed_tokens` row), while observation/reward/done are `type: "text"` (value → string via `format` → tokenizer). Whole-step template `"<action={action},{observation},{reward},{done}>"`. Reward `0.0` and done `0` use `skip` so those text fragments are omitted (commas stay). Heads read the last token of each step (no learnable scratch tokens).

`type: "image"` is also supported when `pretrained` is a vision-language checkpoint (not used in this notebook).


In [ ]:
import torch

from mouse_core.data import (
    DataLoader,
    Augmenter,
    load_stores_from_hub,
)
from mouse_core.objectives import DqnObjective
from mouse_core.models import Model, push_model_to_hub
from mouse_core.models.backbone import Qwen3Backbone
from mouse_core.models.embedding import TextEmbedder
from mouse_core.models.heads import DiscreteActionValueHead


DATASET_ID = "mouse-example-dataset"   # Hugging Face dataset repo for load_stores_from_hub
MODEL_ID = "mouse-example-offline-dqn-text"       # Hugging Face model repo for push_model_to_hub
MAX_ACTIONS = 4                        # number of discrete actions predicted by the head
MAX_OBS_DISCRETE = 64                  # vocabulary size for discrete observations
GRADIENT_STEPS = 20000                 # total optimizer updates
SEQUENCE_LENGTH = 512                  # replay sequence length sampled by DataLoader
BATCH_SIZE = 4                         # sequences per optimizer step


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Load Data

`load_stores_from_hub` downloads the dataset snapshot and reconstructs the saved `Datastore` objects. Each returned store is one ordered environment stream.


In [ ]:
stores = load_stores_from_hub(DATASET_ID, split="train", force_download=True)

## DataLoader And Augmentation

`DataLoader` samples sequence batches from one or more datastores. With `pack=True`, it can combine shorter segments and returns parallel `segment_ids` so the model can isolate attention/RoPE per pack slice and objectives can ignore transitions that cross packed segments.

`next_batch()` returns `(batch, segment_ids)` where `batch` is `list[list[dict]]` shaped `[batch][sequence]` and `segment_ids` is a parallel `list[list[int]]` labeling which pack slice each step came from.

`Augmenter` can transform fields as batches are sampled. A modality spec names a row field and an augmentation type, such as `discrete`, `linear`, or `image`. Use `keep_fields` for values that should pass through unchanged because an objective still needs them. Segment IDs are returned separately from row dicts and are not part of `keep_fields`.


In [ ]:
augmenter = Augmenter(
    modalities=[
        {
            "field": "action",
            "type": "discrete",
            "vocab_size": MAX_ACTIONS,
            "permute": True,
        },
        {
            "field": "observation",
            "type": "discrete",
            "vocab_size": MAX_OBS_DISCRETE,
            "permute": True,
        },
    ],
    keep_fields=["action", "observation", "reward", "done"],
)

loader = DataLoader(
    stores,
    sequence_length=SEQUENCE_LENGTH,
    batch_size=BATCH_SIZE,
    prefetch=4,
    num_workers=0,
    pack=True,
    augmenter=augmenter,
)

## Build The Model

A Mouse Core `Model` has three main pieces:

- `TextEmbedder` converts fields from each step dictionary into model tokens.
- `Qwen3Backbone` processes those tokens with a transformer backbone.
- `DiscreteActionValueHead` predicts one value per discrete action.

`TextEmbedder` modality kinds:

- `text` — turn a value into a string with per-field `format` (e.g. `16` → `"16"`), put it in the step `format`, tokenize
- `token` — integer id selects `embed_tokens[id]` directly (e.g. `16` → the 16th embedding; exactly one token). No per-field `format`.
- `image` — `{field}` inserts a vision span (requires a vision-language checkpoint)

There is no learnable scratch modality on `TextEmbedder` (use `NumericEmbedder` for that). Heads read the last token of each step via `step_token_indices`.

Step template (order is left-to-right in this string):

```text
format="<action={action},{observation},{reward},{done}>"
```

Optional `skip` omits that field's fragment/span; step-format literals such as commas are kept. `col_values` dtypes come from the batch data. Steps pack into a flat `[B, L, D]` sequence.

The backbone exposes `hidden_dim`, and the embedder and head use that same value so the pieces connect cleanly.

### What packed steps look like

Three steps (`action` = `token`, others = `text`; reward/done skipped when zero):

```text
step 0:  action=0, observation=1, reward=0.0, done=0
step 1:  action=2, observation=5, reward=0.0, done=0
step 2:  action=1, observation=7, reward=1.0, done=1
```

Each step fills `format="<action={action},{observation},{reward},{done}>"`.
`{action}` is one vocab embedding; the rest is one tokenized string.
`*` marks the last token of the step — the head reads it to score the **next** action.

**Step 0** → `"<action="` + `embed[0]` + `",observation=1,,>"`

```text
<action=⟦embed 0⟧,observation=1,,>*
```

**Step 1** → `"<action="` + `embed[2]` + `",observation=5,,>"`

```text
<action=⟦embed 2⟧,observation=5,,>*
```

**Step 2** → `"<action="` + `embed[1]` + `",observation=7,reward=1.0,done=1>"`

```text
<action=⟦embed 1⟧,observation=7,reward=1.0,done=1>*
```

Packed in order: step0 · step1 · step2. `step_token_indices` points at each `*`.


In [ ]:
# Same pretrained repo for backbone weights and TextEmbedder tokenizer/embed_tokens.
PRETRAINED = "Qwen/Qwen3-0.6B"
backbone = Qwen3Backbone(pretrained=PRETRAINED)

encoder = TextEmbedder(
    hidden_dim=backbone.hidden_dim,
    pretrained=PRETRAINED,
    format="<action={action},{observation},{reward},{done}>",
    modalities=[
        {"field": "action", "type": "token"},
        {"field": "observation", "type": "text", "format": "observation={observation}"},
        {"field": "reward", "type": "text", "format": "reward={reward}", "skip": 0.0},
        {"field": "done", "type": "text", "format": "done={done}", "skip": 0},
        # Image: put {pixels} in format when pretrained is a vision-language checkpoint.
        # {"field": "pixels", "type": "image"},
    ],
)

head = DiscreteActionValueHead(
    in_features=backbone.hidden_dim,
    out_features=MAX_ACTIONS,
    hidden_dim=backbone.hidden_dim,
    num_layers=1,
    scale=0.1,
)

model = Model(encoder=encoder, backbone=backbone, heads=head).train().to(device)
model.backbone.model.compile()  # optional compile step for faster repeated forwards
print(model)


## Train

The offline training loop is intentionally small because the Mouse Core abstractions do most of the work:

1. `loader.next_batch()` samples step sequences and parallel `segment_ids`.
2. `model(batch, segment_ids=segment_ids)` embeds the dictionaries, runs the backbone with same-segment attention/RoPE, and produces head predictions.
3. `objective(objective_data, predictions)` computes the DQN loss and metrics.
4. The optimizer updates model weights.
5. `model.polyak_update(...)` updates target-network weights used by the objective.

`DqnObjective` interprets the integer `done` field through separate discount factors for normal steps, episode boundaries, and task boundaries. Set those gammas to match the semantics of your environment data.


In [ ]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1.0e-5,
    weight_decay=0.0,
    betas=(0.9, 0.95),
    eps=1.0e-8
)
objective = DqnObjective(
    gamma_step=1.0,                 # discount for ordinary next-step bootstrapping
    gamma_episode_terminal=1.0,     # discount when an episode ends normally
    gamma_episode_truncated=1.0,
    gamma_task_terminal=0.0,        # discount when a task ends normally
    gamma_task_truncated=0.0,
    tau=0.0005,
)

for step in range(GRADIENT_STEPS):

    batch, segment_ids = loader.next_batch()

    predictions, objective_data, _ = model(batch, segment_ids=segment_ids)
    loss, metrics = objective(objective_data, predictions)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    model.polyak_update(action_value_tau=objective.tau)

    if step % 100 == 0:
        print(f"step={step}  loss={loss.item():.4f}  q={metrics['q_values_mean']:.3f}")

loader.close()

## Push To The Hub

`push_model_to_hub` saves the model architecture and weights together. Later, `load_model` can reconstruct the full `Model` without repeating the embedder, backbone, and head definitions.


In [ ]:
model.eval().to("cpu")
url = push_model_to_hub(model, MODEL_ID, private=False, clear=True)
print(f"Pushed to {url}")